In [25]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
import pandas as pd
import fitz  # PyMuPDF

# ============================================================
# CONFIGURATION
# ============================================================
DOC_CODE = "g33a"
INPUT_JSON = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\auto\{DOC_CODE}_content_list copy.json"
INPUT_PDF  = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\SPM\{DOC_CODE}.pdf"
OUTPUT_CSV = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\graphrag_csv\{DOC_CODE}.csv"

# Block types to EXCLUDE
EXCLUDE_TYPES = {"image", "header", "footer", "page_number"}


# ============================================================
# Step 1: Extract text from a single block
# ============================================================
def extract_block_text(block: Dict[str, Any]) -> str:
    """Extract text content from a block based on its type."""
    typ = block.get("type", "")

    if typ == "text":
        return (block.get("text") or "").strip()

    elif typ == "table":
        return (block.get("table_body") or "").strip()

    elif typ == "list":
        items = block.get("list_items") or []
        return "\n".join(str(item).strip() for item in items if str(item).strip())

    elif typ == "code":
        return (block.get("code_body") or "").strip()

    elif typ == "equation":
        return (block.get("text") or "").strip()

    elif typ == "page_footnote":
        return (block.get("text") or "").strip()

    return (block.get("text") or "").strip()


# ============================================================
# Step 2: Extract PDF metadata
# ============================================================
def extract_pdf_metadata(pdf_path: str) -> Dict[str, Any]:
    """Extract metadata from PDF using fitz."""
    try:
        pdf = fitz.open(pdf_path)
        md = pdf.metadata or {}
        pdf.close()
        return {
            "doc_title": Path(pdf_path).stem or None,   # 用檔案名稱作為 doc_title
            "author":  md.get("author") or None,
            "version": md.get("version") or None,
        }
    except Exception as e:
        print(f"Warning: Could not read PDF metadata: {e}")
        return {"doc_title": Path(pdf_path).stem, "author": None, "version": None}


# ============================================================
# Step 3: Aggregate sections by text_level = 1
# ============================================================

def normalize_heading(text: str) -> str:
    """Normalize heading text for deduplication:
    - Remove trailing dots, ellipsis, page numbers
    - Collapse whitespace
    - Lowercase
    """
    text = re.sub(r'[\.…]+\s*\d*\s*$', '', text)  # trailing dots + page nums
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def aggregate_sections(content_list: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Aggregate content by text_level=1 headings.
    Uses last_occurrence to skip TOC duplicate headings:
    - TOC headings appear early (small index)
    - Body headings appear later (large index) → last occurrence = authoritative
    """

    # Pass 1: Find last occurrence index of each heading text
    last_occurrence: Dict[str, int] = {}
    for i, block in enumerate(content_list):
        if block.get("text_level") == 1:
            heading_text = normalize_heading(extract_block_text(block))
            if heading_text:
                last_occurrence[heading_text] = i  # overwrite → last wins

    print(f'  Unique headings found: {len(last_occurrence)}')

    # Pass 2: Aggregate sections, skipping TOC duplicate headings
    sections = []
    current_section_heading = None
    current_parts = []

    for i, block in enumerate(content_list):
        typ = block.get("type", "")
        is_heading = block.get("text_level") == 1

        # Skip excluded types
        if typ in EXCLUDE_TYPES:
            continue

        if is_heading:
            heading_text = normalize_heading(extract_block_text(block))

            # Skip TOC duplicates — only process the last (authoritative) occurrence
            if last_occurrence.get(heading_text) != i:
                continue

            # Save previous section
            if current_section_heading is not None:
                sections.append({
                    "section_title": current_section_heading,
                    "text": "\n\n".join(p for p in current_parts if p),
                })

            # Start new section
            current_section_heading = extract_block_text(block)
            current_parts = [current_section_heading]

        else:
            if current_section_heading is not None:
                text = extract_block_text(block)
                if text.strip():
                    current_parts.append(text)

    # Last section
    if current_section_heading is not None:
        sections.append({
            "section_title": current_section_heading,
            "text": "\n\n".join(p for p in current_parts if p),
        })

    return sections


# ============================================================
# Step 4: Main execution
# ============================================================

# Load content list
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    content_list = json.load(f)
print(f"Loaded {len(content_list)} blocks from {INPUT_JSON}")

# Extract PDF metadata
metadata = extract_pdf_metadata(INPUT_PDF)
print(f"Metadata: {metadata}")

# Aggregate sections
sections = aggregate_sections(content_list)
print(f"Found {len(sections)} sections")

# Build DataFrame
df = pd.DataFrame(sections)
df["doc_title"] = metadata["doc_title"]
df["author"]    = metadata["author"]
df["version"]   = metadata["version"]

# Reorder columns
df = df[["text", "doc_title", "version", "author", "section_title"]]

before = len(df)
df = df[df["text"].str.strip() != df["section_title"].str.strip()]
# Ensure output directory exists
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Saved to: {OUTPUT_CSV}")

display(df.head())

Loaded 310 blocks from C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\g33a\auto\g33a_content_list copy.json
Metadata: {'doc_title': 'g33a', 'author': None, 'version': None}
  Unique headings found: 27
Found 27 sections
✅ Saved to: C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\graphrag_csv\g33a.csv


,text,doc_title,version,author,section_title
0,1. Introduction\n\n1.1 The current HKMA Guidel...,g33a,None,None,1. Introduction
1,2. Customer acceptance policy\n\n2.1 This is a...,g33a,None,None,2. Customer acceptance policy
2,3. Customer due diligence\n\n3.1 This section ...,g33a,None,None,3. Customer due diligence
3,4. Corporate customers\n\n4.1 This section sup...,g33a,None,None,4. Corporate customers
4,5. Trust and nominee accounts\n\n5.1 This sect...,g33a,None,None,5. Trust and nominee accounts


In [27]:
import pandas as pd

csv_path = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\graphrag_csv\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022.csv"

df = pd.read_csv(csv_path)
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
display(df)

Total rows: 26
Columns: ['text', 'doc_title', 'version', 'author', 'section_title']


,text,doc_title,version,author,section_title
0,1 Introduction\n\n1.1 The Personal Data Protec...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,1 Introduction
1,2 Overview of the PDPA\n\n2.1 The PDPA governs...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,2 Overview of the PDPA
2,3 Definitions and related matters\n\n3.1 Befor...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,3 Definitions and related matters
3,4 Individuals\n\n4.1 The PDPA defines an indiv...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,4 Individuals
4,5 Personal data\n\n5.1 Personal data is define...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,5 Personal data
5,6 Organisations\n\n6.1 The PDPA defines an org...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,6 Organisations
6,"7 Collection, Use and Disclosure\n\n7.1 Part 4...",Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,"7 Collection, Use and Disclosure"
7,8 Purposes\n\n8.1 The PDPA does not define the...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,8 Purposes
8,9 Reasonableness\n\n9.1 A number of provisions...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,9 Reasonableness
9,10 Overview of the Data Protection Provisions\...,Advisory Guidelines on Key Concepts in the PDP...,NaN,NaN,10 Overview of the Data Protection Provisions
